# 📖 OCR Pipeline: PDF → Markdown (Gemini Vision)

Pipeline gồm 2 bước:
1. **OCR**: PDF → Markdown thô (Gemini vision đọc ảnh từng trang)
2. **Clean**: Markdown thô → Markdown phân cấp heading chuẩn

**Cải thiện so với bản cũ:**
- Retry logic khi API lỗi
- Rate limiting tránh bị block
- Resume từ trang cuối nếu bị gián đoạn
- Smart chunking theo heading thay vì dấu câu
- System prompt tối ưu cho SGK

## 0. Cài đặt thư viện

In [1]:
# Chỉ chạy lần đầu
# !pip install google-generativeai PyMuPDF pillow python-dotenv

## 1. Import & Cấu hình

In [6]:
import os
import re
import time
from pathlib import Path
from typing import List, Optional

import fitz  # PyMuPDF
from PIL import Image
import google.generativeai as genai

# === API Key ===
# Cách 1: Từ .env
from dotenv import load_dotenv
load_dotenv(Path("../.env"))
API_KEY = os.getenv("GENAI_API_KEY", "")

# Cách 2: Hardcode (không khuyến khích)
# API_KEY = "your-api-key-here"

genai.configure(api_key=API_KEY)

# === Cấu hình ===
OCR_MODEL = "gemini-2.5-flash"        # Model cho OCR (vision)
CLEAN_MODEL = "gemini-2.5-flash"       # Model cho phân cấp heading
DPI = 200                              # Độ phân giải render PDF
TEMP_DIR = "temp_images"               # Thư mục ảnh tạm

DELAY_BETWEEN_PAGES = 2.0              # Giây giữa mỗi trang OCR
DELAY_BETWEEN_CHUNKS = 1.5             # Giây giữa mỗi chunk clean
MAX_RETRIES = 3                        # Số lần retry khi lỗi
RETRY_DELAY = 10.0                     # Giây chờ khi retry
MAX_CHUNK_CHARS = 9000                 # Max chars mỗi chunk clean

os.makedirs(TEMP_DIR, exist_ok=True)
print(f"✅ API Key: {'Đã set' if API_KEY else '❌ CHƯA SET'}")
print(f"✅ OCR Model: {OCR_MODEL}")
print(f"✅ Clean Model: {CLEAN_MODEL}")

✅ API Key: Đã set
✅ OCR Model: gemini-2.5-flash
✅ Clean Model: gemini-2.5-flash


## 2. System Prompts

In [17]:
OCR_SYSTEM_PROMPT = """\
Bạn là chuyên gia trích xuất văn bản từ sách giáo khoa Tin học THPT Việt Nam.

=== NHIỆM VỤ ===
Trích xuất CHỈ nội dung văn bản từ ảnh trang sách, xuất ra dạng Markdown sạch.

=== QUY TẮC BẮT BUỘC ===

1. GIỮ NGUYÊN NỘI DUNG GỐC:
   - Giữ nguyên 100% nội dung văn bản, không tóm tắt, không diễn giải lại
   - Giữ nguyên thuật ngữ chuyên ngành, tên riêng, ví dụ cụ thể
   - Giữ nguyên câu hỏi, bài tập, phần "Tóm tắt bài học", phần "Em cần chú ý"

2. BỎ QUA HOÀN TOÀN (KHÔNG đề cập):
   - Hình ảnh, ảnh minh họa, ảnh chụp màn hình
   - Bảng biểu, biểu đồ, sơ đồ
   - Icon, logo, hình trang trí
   - Số trang, header/footer lặp lại
   - Watermark, dòng chữ "Đọc sách tại hoc10.vn" hoặc tương tự
   - Chú thích hình ảnh (caption dạng "Hình 1.1", "Ảnh minh họa...")

3. XỬ LÝ CODE/MÃ NGUỒN:
   - KHÔNG giữ code gốc
   - Thay bằng mô tả ngắn gọn về chức năng và ý nghĩa của đoạn mã
   - Giữ lại kết quả output nếu có

4. ĐỊNH DẠNG MARKDOWN:
   - Tiêu đề bài lớn: # Bài X: TÊN BÀI
   - Mục chính: ## Tên mục
   - Mục con: ### Tên mục con
   - Nội dung: dùng * cho gạch đầu dòng
   - In đậm cho khái niệm quan trọng: **khái niệm**

5. KHÔNG THÊM:
   - Không thêm lời giới thiệu hay nhận xét cá nhân
   - Không thêm "Trang X" vào output
   - Chỉ xuất Markdown thuần túy
"""

CLEAN_SYSTEM_PROMPT = """\
Bạn là chuyên gia phân cấp và chuẩn hóa tài liệu Markdown từ sách giáo khoa Tin học THPT.

=== NHIỆM VỤ ===
Phân cấp heading, chuẩn hóa cấu trúc, và dọn rác cho đoạn Markdown được cung cấp.

=== QUY TẮC BẮT BUỘC ===

1. GIỮ NGUYÊN 100% NỘI DUNG VĂN BẢN:
   - KHÔNG tóm tắt, KHÔNG xóa, KHÔNG diễn giải lại bất kỳ dòng nào
   - Giữ nguyên tất cả câu hỏi, bài tập, tóm tắt, ví dụ

2. PHÂN CẤP HEADING NỘI DUNG CHÍNH:
   - # cho Chủ đề hoặc Bài lớn (VD: # Bài 1: DỮ LIỆU VÀ THÔNG TIN)
   - ## cho Mục chính trong bài (VD: ## 1. Nguồn thông tin và dữ liệu)
   - ### cho Mục con (VD: ### a) Từ thông tin thành dữ liệu)
   - #### trở đi cho tiểu mục nhỏ hơn

3. CHUẨN HÓA CÁC PHẦN ĐẶC BIỆT CUỐI BÀI (luôn dùng ##):
   - Phần tóm tắt → ## Tóm tắt bài học (dù gốc là ###, ####, hay không có heading)
   - Phần lưu ý → ## Em cần chú ý (dù gốc là ###, ####, hay không có heading)
   - Phần câu hỏi/bài tập luyện tập cuối bài → ## Luyện tập
   - Phần vận dụng → ## Vận dụng
   Tất cả các phần trên PHẢI ở cấp ## (ngang hàng với các mục nội dung chính)

4. THÊM HEADING CHO CÂU HỎI/BÀI TẬP THIẾU HEADING:
   - Nếu phần cuối bài có các câu hỏi dạng "Bài 1.", "Câu 1.", "**Bài 1**" mà KHÔNG có heading phía trước
   - → Thêm ## Luyện tập ngay trước câu hỏi đầu tiên
   - Nhận diện bài tập bằng: dòng bắt đầu với "Bài X.", "Câu X.", "**Bài X**", "* **Bài X**", "* **Câu X**"

5. XÓA RÁC:
   - Xóa: "--- Trang X ---", watermark, header/footer lặp lại
   - Xóa: "Đọc sách tại hoc10.vn", caption hình ảnh còn sót
   - Xóa: comment HTML (<!-- PAGE X -->, <!-- OCR FAILED -->)

6. KHÔNG THÊM gì ngoài heading chuẩn hóa. Trả về Markdown sạch, không giải thích.
"""


print("✅ System prompts đã sẵn sàng")

✅ System prompts đã sẵn sàng


## 3. Hàm OCR (Bước 1: PDF → Markdown thô)

In [8]:
def ocr_single_page(model, img_path: str, page_num: int) -> Optional[str]:
    """OCR 1 trang với retry logic."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            image = Image.open(img_path)
            response = model.generate_content([
                "Trích xuất toàn bộ văn bản tiếng Việt từ ảnh trang sách giáo khoa Tin học này.",
                image
            ])

            page_text = ""
            if hasattr(response, "parts") and response.parts:
                for part in response.parts:
                    if hasattr(part, "text"):
                        page_text += part.text

            if page_text.strip():
                return page_text.strip()
            else:
                print(f"  ⚠️  Trang {page_num}: response rỗng (attempt {attempt}/{MAX_RETRIES})")

        except Exception as e:
            print(f"  ⚠️  Trang {page_num}: lỗi '{str(e)[:60]}' (attempt {attempt}/{MAX_RETRIES})")
            if attempt < MAX_RETRIES:
                print(f"       Retry sau {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)

    return None


def ocr_pdf(pdf_path: str, output_path: str, start_page: int, end_page: int):
    """
    OCR từng trang PDF → ghi ra Markdown.
    Hỗ trợ resume: detect trang cuối đã OCR và tiếp tục.
    """
    model = genai.GenerativeModel(
        model_name=OCR_MODEL,
        system_instruction=OCR_SYSTEM_PROMPT
    )

    doc = fitz.open(pdf_path)
    total_pages = doc.page_count
    end_page = min(end_page, total_pages)

    print(f"📄 PDF: {pdf_path}")
    print(f"📑 Tổng trang: {total_pages} | OCR: trang {start_page+1} → {end_page}")

    # === Resume detection ===
    resume_page = start_page
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            existing = f.read()
        page_markers = re.findall(r"<!-- PAGE (\d+) -->", existing)
        if page_markers:
            last_done = max(int(p) for p in page_markers)
            resume_page = last_done
            print(f"⏩ Resume: đã OCR đến trang {last_done}, tiếp tục từ trang {resume_page+1}")

    if resume_page >= end_page:
        print("✅ Tất cả các trang đã được OCR!")
        return

    # === OCR từng trang ===
    success = 0
    fail = 0

    with open(output_path, "a", encoding="utf-8") as f_out:
        for page_idx in range(resume_page, end_page):
            page_num = page_idx + 1

            # Render trang → ảnh
            page = doc.load_page(page_idx)
            pix = page.get_pixmap(dpi=DPI)
            img_path = os.path.join(TEMP_DIR, f"page_{page_num}.png")
            pix.save(img_path)

            # Gọi OCR
            page_text = ocr_single_page(model, img_path, page_num)

            if page_text:
                f_out.write(f"<!-- PAGE {page_idx} -->\n")
                f_out.write(page_text + "\n\n")
                f_out.flush()
                success += 1
                print(f"  ✅ Trang {page_num}/{end_page} — {len(page_text)} chars")
            else:
                fail += 1
                f_out.write(f"<!-- PAGE {page_idx} -->\n")
                f_out.write(f"<!-- OCR FAILED: trang {page_num} -->\n\n")
                f_out.flush()
                print(f"  ❌ Trang {page_num}/{end_page} — FAILED")

            # Xóa ảnh tạm
            try:
                os.remove(img_path)
            except OSError:
                pass

            # Rate limit
            if page_idx < end_page - 1:
                time.sleep(DELAY_BETWEEN_PAGES)

    doc.close()
    print(f"\n🎉 OCR xong! ✅ {success} trang | ❌ {fail} trang | 📝 {output_path}")


print("✅ Hàm OCR đã sẵn sàng")

✅ Hàm OCR đã sẵn sàng


## 4. Hàm Clean (Bước 2: Markdown thô → Markdown sạch)

In [9]:
def smart_chunk(text: str, max_chars: int = MAX_CHUNK_CHARS) -> List[str]:
    """Chia text theo heading # thay vì dấu câu."""
    sections = re.split(r"(?=^#{1,2}\s)", text, flags=re.MULTILINE)
    sections = [s.strip() for s in sections if s.strip()]

    chunks = []
    current = ""

    for section in sections:
        if len(current) + len(section) + 2 <= max_chars:
            current += "\n\n" + section if current else section
        else:
            if current:
                chunks.append(current.strip())
            if len(section) > max_chars:
                # Section quá dài → cắt theo paragraph
                paras = section.split("\n\n")
                sub = ""
                for p in paras:
                    if len(sub) + len(p) + 2 <= max_chars:
                        sub += "\n\n" + p if sub else p
                    else:
                        if sub:
                            chunks.append(sub.strip())
                        sub = p
                if sub.strip():
                    chunks.append(sub.strip())
                current = ""
            else:
                current = section

    if current.strip():
        chunks.append(current.strip())

    return chunks if chunks else [text]


def clean_single_chunk(model, chunk: str) -> Optional[str]:
    """Clean 1 chunk với retry."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content([
                "Phân cấp lại heading cho đoạn Markdown sau, giữ nguyên nội dung:",
                chunk
            ])

            result = ""
            if hasattr(response, "parts") and response.parts:
                for part in response.parts:
                    if hasattr(part, "text"):
                        result += part.text

            if result.strip():
                return result.strip()

        except Exception as e:
            print(f"\n    ⚠️ Lỗi: {str(e)[:50]} (attempt {attempt})")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

    return None


def post_process(text: str) -> str:
    """Xóa các dòng rác phổ biến."""
    patterns = [
        r"Đọc sách tại hoc10\.vn\s*",
        r"Nguồn:?\s*hoc10\.vn\s*",
        r"```\s*```",
    ]
    for p in patterns:
        text = re.sub(p, "\n", text, flags=re.IGNORECASE)

    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_markdown(input_path: str, output_path: str):
    """Phân cấp lại heading và làm sạch Markdown."""
    model = genai.GenerativeModel(
        model_name=CLEAN_MODEL,
        system_instruction=CLEAN_SYSTEM_PROMPT
    )

    with open(input_path, "r", encoding="utf-8") as f:
        raw_md = f.read()

    # Xóa markers
    raw_md = re.sub(r"<!-- PAGE \d+ -->\n?", "", raw_md)
    raw_md = re.sub(r"<!-- OCR FAILED: trang \d+ -->\n?", "", raw_md)

    chunks = smart_chunk(raw_md)
    print(f"📦 Input: {input_path} ({len(raw_md)} chars) → {len(chunks)} chunks")

    results = []
    for i, chunk in enumerate(chunks):
        print(f"  ⚙️  Chunk {i+1}/{len(chunks)} ({len(chunk)} chars)...", end=" ")

        cleaned = clean_single_chunk(model, chunk)
        if cleaned:
            results.append(cleaned)
            print(f"✅")
        else:
            results.append(chunk)
            print("⚠️ giữ nguyên")

        if i < len(chunks) - 1:
            time.sleep(DELAY_BETWEEN_CHUNKS)

    final = post_process("\n\n".join(results))

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(final)

    print(f"\n🎉 Clean xong! → {output_path} ({len(final)} chars)")


print("✅ Hàm Clean đã sẵn sàng")

✅ Hàm Clean đã sẵn sàng


---
## 5. 🚀 CHẠY OCR (Bước 1)

Sửa các biến dưới đây cho phù hợp với file PDF muốn OCR.

In [14]:
# ========================================
# 🔧 SỬA CÁC BIẾN NÀY
# ========================================

PDF_PATH = "../KNTT/SGK TIN HOC 11 ICT - KNTT.pdf"   # Đường dẫn PDF
START_PAGE = 5           # Trang bắt đầu (0-indexed, bỏ bìa/mục lục)
END_PAGE = 6       # Trang kết thúc (exclusive)
RAW_OUTPUT = "../RawData/SGK_cc.md"   # File output Markdown thô

# ========================================
# Chạy OCR
# ========================================
ocr_pdf(
    pdf_path=PDF_PATH,
    output_path=RAW_OUTPUT,
    start_page=START_PAGE,
    end_page=END_PAGE
)

📄 PDF: ../KNTT/SGK TIN HOC 11 ICT - KNTT.pdf
📑 Tổng trang: 152 | OCR: trang 6 → 6
⏩ Resume: đã OCR đến trang 4, tiếp tục từ trang 5
  ✅ Trang 5/6 — 2002 chars
  ✅ Trang 6/6 — 1915 chars

🎉 OCR xong! ✅ 2 trang | ❌ 0 trang | 📝 ../RawData/SGK_cc.md


## 6. 🧹 CHẠY CLEAN (Bước 2)

Chạy sau khi bước OCR xong. Phân cấp lại heading Markdown.

In [24]:
# ========================================
# 🔧 SỬA CÁC BIẾN NÀY
# ========================================

RAW_INPUT = "../RawData/SGK_Tin10_KNTT.md"          # File từ bước OCR
CLEAN_OUTPUT = "../RawData/SGK_Tin10_KNTT_clean.md"  # File output sạch

# ========================================
# Chạy Clean
# ========================================
clean_markdown(
    input_path=RAW_INPUT,
    output_path=CLEAN_OUTPUT
)

📦 Input: ../RawData/SGK_Tin10_KNTT.md (286845 chars) → 37 chunks
  ⚙️  Chunk 1/37 (6818 chars)... ✅
  ⚙️  Chunk 2/37 (5034 chars)... ✅
  ⚙️  Chunk 3/37 (7223 chars)... ✅
  ⚙️  Chunk 4/37 (6749 chars)... ✅
  ⚙️  Chunk 5/37 (8667 chars)... ✅
  ⚙️  Chunk 6/37 (8710 chars)... ✅
  ⚙️  Chunk 7/37 (8875 chars)... ✅
  ⚙️  Chunk 8/37 (8618 chars)... ✅
  ⚙️  Chunk 9/37 (8348 chars)... ✅
  ⚙️  Chunk 10/37 (7546 chars)... ✅
  ⚙️  Chunk 11/37 (7720 chars)... ✅
  ⚙️  Chunk 12/37 (6415 chars)... ✅
  ⚙️  Chunk 13/37 (8319 chars)... ✅
  ⚙️  Chunk 14/37 (5939 chars)... ✅
  ⚙️  Chunk 15/37 (7658 chars)... ✅
  ⚙️  Chunk 16/37 (7971 chars)... ✅
  ⚙️  Chunk 17/37 (8218 chars)... ✅
  ⚙️  Chunk 18/37 (7706 chars)... ✅
  ⚙️  Chunk 19/37 (8901 chars)... ✅
  ⚙️  Chunk 20/37 (6147 chars)... ✅
  ⚙️  Chunk 21/37 (8736 chars)... ✅
  ⚙️  Chunk 22/37 (8633 chars)... ✅
  ⚙️  Chunk 23/37 (8723 chars)... ✅
  ⚙️  Chunk 24/37 (7982 chars)... ✅
  ⚙️  Chunk 25/37 (8193 chars)... ✅
  ⚙️  Chunk 26/37 (7953 chars)... ✅
  ⚙️  Ch

## 7. 🔍 Kiểm tra kết quả

Xem nhanh vài dòng đầu của file output.

In [8]:
# Xem file thô
output_file = "../RawData/SGK_Tin12_CD.md"  # Đổi thành file muốn xem

if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    lines = content.split("\n")
    print(f"📝 File: {output_file}")
    print(f"📊 Tổng: {len(content)} chars | {len(lines)} dòng")
    print(f"\n{'─' * 50}")
    print("\n".join(lines[:50]))  # 50 dòng đầu
    print(f"\n{'─' * 50}")
    print(f"... (còn {len(lines) - 50} dòng)")
else:
    print(f"❌ File chưa tồn tại: {output_file}")

📝 File: ../RawData/SGK_Tin12_CD.md
📊 Tổng: 0 chars | 1 dòng

──────────────────────────────────────────────────


──────────────────────────────────────────────────
... (còn -49 dòng)


## 8. 🧪 Test OCR 1 trang (Debug)

Test nhanh 1 trang để kiểm tra prompt có ổn không trước khi chạy cả cuốn.

In [9]:
# === Test nhanh 1 trang ===
TEST_PDF = "../CanhDieu/SGK TIN HOC 12 CS - CANH DIEU.pdf"
TEST_PAGE = 10  # Trang muốn test (0-indexed)

model = genai.GenerativeModel(
    model_name=OCR_MODEL,
    system_instruction=OCR_SYSTEM_PROMPT
)

doc = fitz.open(TEST_PDF)
page = doc.load_page(TEST_PAGE)
pix = page.get_pixmap(dpi=DPI)
test_img_path = os.path.join(TEMP_DIR, "test_page.png")
pix.save(test_img_path)
doc.close()

print(f"🖼️ Đã render trang {TEST_PAGE + 1}")

# OCR
result = ocr_single_page(model, test_img_path, TEST_PAGE + 1)

if result:
    print(f"\n✅ OCR thành công ({len(result)} chars)")
    print("═" * 50)
    print(result)
    print("═" * 50)
else:
    print("❌ OCR thất bại")

# Cleanup
try:
    os.remove(test_img_path)
except:
    pass

🖼️ Đã render trang 11

✅ OCR thành công (1364 chars)
══════════════════════════════════════════════════
hát mới. Ví dụ về **AI tạo sinh** trong lĩnh vực này là các hệ thống Mubert, Beatoven,... **AI tạo sinh hình ảnh** giúp máy tính có khả năng vẽ tranh theo mô tả yêu cầu, ví dụ Midjourney, DALL-E,...

Em hãy cho biết mỗi phát biểu sau về AI là đúng hay sai:
a) “**Turing Test**” là bài kiểm tra trí tuệ của máy tính.
b) Nhờ mở rộng phạm vi ứng dụng mà **AI yếu** phát triển thành **AI mạnh**.
c) AI tạo sinh có thể giúp học sinh viết được một bài văn tả cảnh đẹp của quê hương.
d) AI có thể tự hành động một cách hợp lí.

Năm 1997, máy tính **Deep Blue** của IBM đánh bại Đại kiện tướng cờ vua **Garry Kasparov**. Đây là lần đầu tiên một chương trình máy tính đánh bại một nhà vô địch thế giới về cờ vua. Em hãy giải thích vì sao sự kiện đó được xem là một thành tựu của trí tuệ nhân tạo.

Câu 1. AI là gì? **AI mạnh** là gì? **AI yếu** là gì?
Câu 2. Lĩnh vực nghiên cứu nào giúp máy tính có khả n